# BitirmeProject — Gemma3-4B QLoRA Fine-tune (v2)

**Dataset:** 1417 örnek (487 agent + 364 scaffold + 308 enrich + 258 plan)  
**Donanım:** Colab Free T4 16 GB — `Runtime → Change runtime type → T4 GPU`  
**Beklenen süre:** 3-4 saat (240 step × ~50s)  
**Çıktı:** `MyDrive/bp-finetune-v2/adapters/gemma3-4b-bp-v2/` LoRA adapter

## v1'den farklılıklar (geçen seferki hataları önlemek için)

| Sorun (v1) | Çözüm (v2) |
|---|---|
| 125 step'te kesildi | `save_steps=50`, `save_total_limit=10` — daha sık checkpoint |
| Disconnect'te kayıp | Resume logic çalışıyor + her zaman son checkpoint'ten devam |
| 0 tool-calling örneği | 487 agent örneği eklendi |
| LoRA r=8/16 yetersiz | r=32 (kapasite 2×) |
| `save_pretrained_gguf` Gemma3 multimodal'da kırık | **GGUF export YEREL'de** yapılacak (`manual_merge.py` + llama.cpp) — notebook'ta yok |
| Dependency çakışmaları | Tüm versiyonlar pinned aşağıda |
| Colab RAM 12 GB aşımı | GGUF burada yapılmayacak; adapter ~50MB kalır |

## Akış
1. GPU + dependency kontrolü
2. Drive mount + dataset doğrulama
3. Bağımlılıkları yükle (pinned)
4. Model + LoRA yükle
5. Dataset hazırla (chat template)
6. Train (resume_from_checkpoint=True ile)
7. Final adapter'ı Drive'a kaydet
8. Quick sanity test (5 örnek)

**Disconnect olursa:** Cell 1-5'i tekrar çalıştır → Cell 6'da `last_ckpt` otomatik bulunur → eğitim kaldığı yerden devam eder.

## 1) GPU + sistem kontrolü

Beklenen: Tesla T4, 16 GB VRAM, ~12 GB sistem RAM.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free,driver_version --format=csv
!free -h | head -2
!df -h / | head -2
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'GPU yok — Runtime → Change runtime type → T4 GPU seç'

## 2) Drive mount + dataset doğrulama

Drive'da olması gereken yapı:
```
MyDrive/bp-finetune-v2/
├── datasets/
│   ├── train-v2.train.jsonl   (1275 örnek, ~3.9 MB)
│   └── train-v2.val.jsonl     (142 örnek, ~440 KB)
├── configs/
│   └── v2.yaml
└── eval/                       (opsiyonel, smoke test için)
    └── samples.jsonl
```

Eğer eksik klasör/dosya varsa, hücre stop edip net mesaj verir.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, yaml, json

BASE = '/content/drive/MyDrive/bp-finetune-v2'
TRAIN_PATH = f'{BASE}/datasets/train-v2.train.jsonl'
VAL_PATH = f'{BASE}/datasets/train-v2.val.jsonl'
CONFIG_PATH = f'{BASE}/configs/v2.yaml'
ADAPTER_OUT = f'{BASE}/adapters/gemma3-4b-bp-v2'
CKPT_DIR = f'{BASE}/checkpoints-v2'

for d in (ADAPTER_OUT, CKPT_DIR, f'{BASE}/adapters'):
    os.makedirs(d, exist_ok=True)

missing = [p for p in (TRAIN_PATH, VAL_PATH, CONFIG_PATH) if not os.path.exists(p)]
if missing:
    raise FileNotFoundError('Drive\'da eksik dosyalar:\n  ' + '\n  '.join(missing) +
                            '\n\nv2-drive-bundle/ klasörünü MyDrive/bp-finetune-v2/ altına yükledin mi?')

with open(CONFIG_PATH, encoding='utf-8') as f:
    CFG = yaml.safe_load(f)

# Dataset boyut sanity check
with open(TRAIN_PATH, encoding='utf-8') as f:
    train_count = sum(1 for _ in f)
with open(VAL_PATH, encoding='utf-8') as f:
    val_count = sum(1 for _ in f)

print(f'[config] {CFG["version"]} | base: {CFG["base_model"]["name"]}')
print(f'[lora] r={CFG["lora"]["r"]} alpha={CFG["lora"]["alpha"]}')
print(f'[train] {train_count} örnek | val: {val_count}')
print(f'[epochs] {CFG["training"]["num_train_epochs"]} × ~{train_count // 16} step/epoch = ~{(train_count // 16) * CFG["training"]["num_train_epochs"]} toplam step')
assert train_count > 1000, f'train_count {train_count} beklenenden az'
assert val_count > 100, f'val_count {val_count} beklenenden az'

## 3) Bağımlılıklar — pinned versiyonlar

v1'de yaşanan çakışmaları önlemek için her şey pin'li.  
**Önemli:** `--no-deps` flag'leri kasıtlı — Unsloth/trl/peft kendi torch versiyonunu zorlayabilir, biz Colab'ın CUDA-uyumlu torch'unu koruyoruz.  
Cell çalıştıktan sonra **runtime restart YOK** — Colab session korunur.

In [ ]:
%%capture
# Pip cache temizle + güncelle
!pip install -U pip

# Unsloth (Colab T4 için resmi flavor) — kendi CUDA-uyumlu torch'unu kullanır
!pip install --upgrade 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'

# Eğitim stack'i — versiyonlar Unsloth+T4 kombinasyonu için test edildi
!pip install --no-deps 'xformers<0.0.27' 'trl<0.9.0' peft==0.13.2 accelerate==1.0.1 bitsandbytes==0.44.1

# Yardımcılar
!pip install datasets==3.0.1 pyyaml==6.0.2 sentencepiece protobuf

In [ ]:
# Version doğrulama — sürpriz olmasın
import torch, transformers, peft, trl, datasets, bitsandbytes
try:
    import unsloth
    print('unsloth', unsloth.__version__)
except Exception as e:
    print('unsloth import error:', e)
print('torch        ', torch.__version__)
print('transformers ', transformers.__version__)
print('peft         ', peft.__version__)
print('trl          ', trl.__version__)
print('datasets     ', datasets.__version__)
print('bitsandbytes ', bitsandbytes.__version__)
print('cuda         ', torch.version.cuda, '| device:', torch.cuda.get_device_name(0))

## 4) Model + LoRA

`unsloth/gemma-3-4b-it-bnb-4bit` — 4-bit quantized, ~5 GB VRAM kullanır.  
LoRA r=32, alpha=64 (v2.yaml'dan).

In [ ]:
from unsloth import FastLanguageModel

bm = CFG['base_model']
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=bm['name'],
    max_seq_length=bm['max_seq_length'],
    dtype=None,                       # T4 → fp16 otomatik
    load_in_4bit=bm['load_in_4bit'],
)

lc = CFG['lora']
model = FastLanguageModel.get_peft_model(
    model,
    r=lc['r'],
    lora_alpha=lc['alpha'],
    lora_dropout=lc['dropout'],
    bias=lc['bias'],
    target_modules=lc['target_modules'],
    use_gradient_checkpointing=lc['use_gradient_checkpointing'],
    random_state=lc['random_state'],
)

model.print_trainable_parameters()
print(f'\n[gpu] VRAM kullanımı: {torch.cuda.memory_allocated() / 1024**3:.2f} GB / {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB')

## 5) Dataset + chat template

Her örnek `messages: [{role, content}, ...]` formatında.  
Agent örneklerinde multi-turn (5-9 mesaj), diğerlerinde 3 mesaj.  
`apply_chat_template` Gemma3 chat template'ini uygular → trainer flat text görür.

In [ ]:
from datasets import load_dataset

ds = load_dataset('json', data_files={'train': TRAIN_PATH, 'validation': VAL_PATH})
print(ds)

def _format(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

ds = ds.map(_format, remove_columns=[c for c in ds['train'].column_names if c != 'text'])

# Sanity check — agent örneğinin chat template'i nasıl görünüyor?
import json
for i in range(len(ds['train'])):
    sample = ds['train'][i]
    if '[tool]' in sample['text']:                   # agent multi-turn
        print('=== AGENT örnek (chat template uygulanmış) ===')
        print(sample['text'][:1200])
        print('... (kısaltıldı)')
        break
print('\n=== feature başına örnek sayıları ===')
# train-v2.stats.json'dan zaten biliyoruz ama hızlı confirm:
print(f"train toplam: {len(ds['train'])}, val toplam: {len(ds['validation'])}")

## 6) Train — checkpoint'lerle

**Kritik:** Resume logic her zaman çalışır. Disconnect olursa:
1. Notebook'u baştan aç
2. Cell 1-5'i sırayla çalıştır
3. **Bu cell** otomatik olarak son checkpoint'i bulup oradan devam eder

**Çıktı:** Her 50 step'te Drive'a flush + 10 checkpoint tutulur.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

tc = CFG['training']
args = TrainingArguments(
    output_dir=CKPT_DIR,
    num_train_epochs=tc['num_train_epochs'],
    per_device_train_batch_size=tc['per_device_train_batch_size'],
    gradient_accumulation_steps=tc['gradient_accumulation_steps'],
    learning_rate=tc['learning_rate'],
    lr_scheduler_type=tc['lr_scheduler_type'],
    warmup_ratio=tc['warmup_ratio'],
    weight_decay=tc['weight_decay'],
    optim=tc['optim'],
    fp16=tc['fp16'],
    bf16=tc['bf16'],
    logging_steps=tc['logging_steps'],
    save_steps=tc['save_steps'],
    save_total_limit=tc['save_total_limit'],
    eval_steps=tc['eval_steps'],
    evaluation_strategy=tc['evaluation_strategy'],
    report_to=tc['report_to'],
    seed=tc['seed'],
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds['train'],
    eval_dataset=ds['validation'],
    dataset_text_field='text',
    max_seq_length=CFG['base_model']['max_seq_length'],
    packing=CFG['dataset']['packing'],
    args=args,
)

# --- Resume logic ---
last_ckpt = None
if os.path.isdir(CKPT_DIR):
    ckpts = [d for d in os.listdir(CKPT_DIR) if d.startswith('checkpoint-')]
    if ckpts:
        ckpts.sort(key=lambda x: int(x.split('-')[1]))
        last_ckpt = os.path.join(CKPT_DIR, ckpts[-1])
        print(f'[resume] kaldığı yerden devam: {last_ckpt}')
    else:
        print('[resume] checkpoint yok, baştan başla')

print(f'[gpu] eğitim öncesi VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB')
trainer.train(resume_from_checkpoint=last_ckpt)
print('[done] eğitim tamamlandı.')

## 7) Final adapter'ı Drive'a kaydet

Bu hücre **eğitim bittiğinde** ya da **manuel olarak** çalıştırılabilir.  
Adapter ~50 MB — Drive'a kayıt 30 saniye sürer.

In [ ]:
# LoRA adapter (sadece — base model değil)
model.save_pretrained(ADAPTER_OUT)
tokenizer.save_pretrained(ADAPTER_OUT)

print(f'[adapter] {ADAPTER_OUT}')
for f in sorted(os.listdir(ADAPTER_OUT)):
    p = os.path.join(ADAPTER_OUT, f)
    if os.path.isfile(p):
        sz = os.path.getsize(p) / 1024**2
        print(f'  {f}: {sz:.1f} MB')
    else:
        print(f'  {f}/')

print('\n[next] Bu klasörü Drive\'dan indir:')
print(f'   {ADAPTER_OUT}')
print('   → lokalde manual_merge.py + llama.cpp ile GGUF\'a çevir.')
print('   → ollama create bp-agent-v2 -f Modelfile')
print('   → .env: OLLAMA_USE_FINETUNED=true, OLLAMA_FINETUNED_MODEL=bp-agent-v2')

## 8) Quick sanity test (opsiyonel)

5 örnek prompt → model JSON üretiyor mu? Bu, GGUF export öncesi sanity check.  
Tüm prompt'lar Türkçe ve gerçek BitirmeProject domain'inden.

In [ ]:
FastLanguageModel.for_inference(model)

TEST_PROMPTS = [
    # agent: tek-tool read
    {'feat': 'agent', 'system': (
        "Sen BitirmeProject AI agent'ısın. Kullanıcının doğal dilde yazdığı isteği araç çağrılarıyla "
        "gerçekleştirmek için tool catalog'unu kullanırsın. Her adımda ya bir tool çağırırsın ya da "
        "konuşmayı 'final' ile bitirirsin. Yalnızca geçerli JSON döndürürsün; markdown fence, açıklama, "
        "İngilizce sızıntısı yasak. Türkçe yanıt ver. Aynı tool'u gereksiz tekrar çağırma, önce mevcut "
        "durumu sorgulamak için get_active_sprint / get_project_issues kullan.\n\n"
        "Tool catalog (kullanılabilir araçlar):\n"
        '[{"name":"get_active_sprint","description":"Aktif sprint\'i döner.","input_schema":{"type":"object","properties":{}}},'
        '{"name":"get_project_issues","description":"Issue listesini döner.","input_schema":{"type":"object","properties":{}}}]\n\n'
        "Yanıt formatı — yalnızca aşağıdaki iki JSON şemadan birinde yanıt ver. Markdown fence, açıklama metni yasak.\n"
        "1) Tool çağırmak için:\n   {\"tool_calls\": [{\"name\": \"<tool_name>\", \"input\": { ... }}]}\n"
        "2) Konuşmayı bitirmek için:\n   {\"final\": \"<kullanıcıya gönderilecek Türkçe mesaj>\"}"
    ), 'user': 'Aktif sprint hangisi?'},
    # agent: tek-tool write
    {'feat': 'agent', 'system': '...', 'user': "'Login sayfasında 500 hatası' başlıklı High priority issue oluştur"},
    # scaffold
    {'feat': 'scaffold-project', 'system': (
        "Sen BitirmeProject AI agent'ısın. Kullanıcının serbest metin talebinden ProjectDraft JSON'u üretirsin. "
        "Yalnızca geçerli JSON döndürürsün; markdown fence, açıklama, İngilizce sızıntısı yasak. Türkçe yanıt ver."
    ), 'user': json.dumps({'description': 'Üniversite mezunları için kariyer takip uygulaması istiyoruz.', 'context': {'area': 'kariyer', 'sub': 'mezun', 'scale': 'küçük (5-20 kullanıcı)'}}, ensure_ascii=False)},
    # enrich
    {'feat': 'enrich-issue', 'system': (
        "Sen BitirmeProject AI agent'ısın. Verilen issue başlığından detaylı issue spesifikasyonu üretirsin "
        "(description, acceptanceCriteria, edgeCases, storyPoints). Yalnızca geçerli JSON döndürürsün. Türkçe."
    ), 'user': json.dumps({'title': 'Şifre sıfırlama e-postası gelmiyor', 'projectContext': 'web app / auth'}, ensure_ascii=False)},
    # plan
    {'feat': 'generate-plan', 'system': (
        "Sen BitirmeProject AI agent'ısın. Mevcut projeye yeni özellik sprint planı üretirsin. "
        "Yalnızca geçerli JSON (sprints[]) döndürürsün. Türkçe."
    ), 'user': json.dumps({'projectName': 'Kariyer Takip', 'description': 'Bildirim sistemi eklenecek.'}, ensure_ascii=False)},
]

qe = CFG['quick_eval']
ok = 0
for i, t in enumerate(TEST_PROMPTS, 1):
    msgs = [{'role': 'system', 'content': t['system']}, {'role': 'user', 'content': t['user']}]
    inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
    gen = model.generate(inputs, max_new_tokens=qe['max_new_tokens'], temperature=qe['temperature'], top_p=qe['top_p'], do_sample=True, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(gen[0][inputs.shape[1]:], skip_special_tokens=True)
    parsed = False
    try:
        s = text.find('{'); e = text.rfind('}')
        if s != -1 and e > s:
            json.loads(text[s:e + 1])
            parsed = True
    except Exception:
        pass
    status = 'OK  ' if parsed else 'FAIL'
    if parsed:
        ok += 1
    print(f'[{i}/{len(TEST_PROMPTS)}] {status} [{t["feat"]:18s}] {t["user"][:60]}...')
    print(f'   → {text[:200]}...\n')

print(f'=== JSON parse: {ok}/{len(TEST_PROMPTS)} ===')
if ok < len(TEST_PROMPTS) // 2:
    print('UYARI: < %50 başarı. Eğitim yeterince ilerlemedi olabilir; loss curve\'a bak.')

## 9) Tamam — sonraki adımlar

**Drive'da hazır:**
- `MyDrive/bp-finetune-v2/adapters/gemma3-4b-bp-v2/` — LoRA adapter (~50 MB)
- `MyDrive/bp-finetune-v2/checkpoints-v2/checkpoint-N/` — son N checkpoint (geri dönüş için)

**Yerelde yapılacaklar** (memory'deki [project_gemma3_export_pipeline]'da kayıtlı):
1. Adapter klasörünü Drive'dan yerele indir
2. `manual_merge.py` ile LoRA → fp16 merged HF
3. `llama.cpp/convert_hf_to_gguf.py` → f16 GGUF
4. `llama-quantize.exe` → Q4_K_M
5. `ollama create bp-agent-v2 -f Modelfile`
6. `.env`: `OLLAMA_USE_FINETUNED=true`, `OLLAMA_FINETUNED_MODEL=bp-agent-v2`
7. `docker compose up -d --force-recreate ai-api`

**Bağlanma:** Eğitim sırasında sekmeyi kapatma. Disconnect olursa cell 1-5 → cell 6 ile devam.

**Süre:** T4'te 240 step × ~50s ≈ **3-3.5 saat**.